# Optimizing PID Controller Gains for a Robotic System to Minimize Error Response
**Case Study: Automated Tuning of a UAV Altitude Controller via Genetic Algorithm**

This notebook demonstrates the application of an evolutionary algorithm to automatically discover the optimal Proportional ($K_p$), Integral ($K_i$), and Derivative ($K_d$) gains for a drone's altitude controller. 

### Step 1: Population Initialization
The Genetic Algorithm (GA) begins by generating a randomized initial population. Each candidate solution in the population is represented by a continuous binary chromosome. The total length of this bitstring is dynamically calculated based on the required resolution (`n_bits`) and the number of variables being optimized within the bounds.

In [1]:
import numpy as np

# 1. Define our rules (bounds used in this case is the search limit for a drone)
bounds = [
    [0.0, 10.0], # Kp
    [0.0, 2.0], # Ki
    [0.0, 2.0], # Kd
]

n_bits = 8 # Number of bits
n_pop = 10 # Number of population

# 2. Calculate how many bits per drone
total_bits_per_drone = n_bits * len(bounds)
print(f'Total bits per drone: {total_bits_per_drone}')

# 3. Create the population
population = []
for _ in range(n_pop):
    drone_chromosome = np.random.randint(0, 2, total_bits_per_drone).tolist()
    population.append(drone_chromosome)

print("Population:")
for i, drone in enumerate(population):
    print(f"Drone {i+1}: {drone}")

Total bits per drone: 24
Population:
Drone 1: [0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0]
Drone 2: [0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0]
Drone 3: [0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0]
Drone 4: [0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0]
Drone 5: [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1]
Drone 6: [1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0]
Drone 7: [1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1]
Drone 8: [0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0]
Drone 9: [1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Drone 10: [0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0]


### Step 2: Chromosome Decoding
To evaluate the fitness of our population, we must translate the binary strings (genotype) back into real-world floating-point numbers (phenotype). This function slices the bitstring into individual variable segments, converts the binary segments into integers, and scales them linearly to fit within our predefined physical boundaries.

In [2]:
def decode(bounds, n_bits, bitstring):
    decoded = []
    largest = 2 ** n_bits

    for i in range(len(bounds)):
        # 1. Slice the 24-bits string into 8-bit chunks
        start = i * n_bits
        end = (i + 1) * n_bits
        substring = bitstring[start:end]
        # print(f'Substring: {substring}')

        # 2. Convert the list of bits into a single integer
        chars = ''.join([str(s) for s in substring])
        integer = int(chars, 2)

        # 3. Scale the integer to fit inside our specific bounds (GA - fx1 NOTES.pdf by Dr. Taniza page 10)
        value = bounds[i][0] + (integer/largest) * (bounds[i][1] - bounds[i][0])
        # print(f'Value: {value}')

        # 4. Add the final value to our decoded list
        decoded.append(value)

    return decoded

### Step 3: Fitness Evaluation (The Objective Function)
This is the core environment where our decoded PID gains are tested. We utilize a lightweight 1D physics simulation to model the drone's vertical thrust towards a 10.0m setpoint over 50 discrete time steps. 

To heavily penalize the controller for large deviations from the target, we calculate the **Integral of Squared Error (ISE)**. Because the GA architecture is designed to *maximize* fitness, we invert the final ISE value so that a smaller flight error yields a higher overall fitness score.

In [3]:
def drone_objective(decoded_values):
    kp = decoded_values[0]
    ki = decoded_values[1]
    kd = decoded_values[2]

    # 1. Setup mock drone environment
    setpoint = 10.0
    altitude = 0.0

    integral_error = 0.0
    previous_error = 0.0
    ise = 0.0 # Integral Squared Error

    # 2. Quick flight simulation for 50 time steps
    for t in range(50):
        # Current error
        error = setpoint - altitude

        # Accumuate integral error
        integral_error += error

        # derivative error
        derivative_error = error - previous_error

        # Core PID equation
        thrust = (kp * error) + (ki * integral_error) + (kd * derivative_error)

        # Mock physics: altitude changes based on our thrust
        altitude += thrust * 0.1

        # ISE: Square the error to heavily penalize being far from the target
        ise += error**2

        # Save current error for the next loop
        previous_error = error

    fitness_score = 1.0 / (1.0 + ise)
    return fitness_score

### Step 4: Tournament Selection
To breed the next generation, we must select strong parents while maintaining genetic diversity to avoid premature convergence on local optima. We utilize **Tournament Selection**: randomly sampling a subset of `k=3` individuals from the population and selecting the individual with the highest fitness score to pass on its genetic material.

In [4]:
def selection(pop, scores, k=3):
    selection_ix = np.random.randint(len(pop))

    for ix in np.random.randint(0, len(pop), k-1):

        if scores[ix] > scores[selection_ix]:
            selection_ix = ix
    
    return pop[selection_ix]

### Step 5: Crossover (Recombination)
Once two parents are selected, they undergo **Single-Point Crossover** based on a probability threshold (`r_cross`). A random locus (index) is chosen along the chromosome, and the parents exchange their genetic tails to produce two novel offspring that inherit traits from both.

In [5]:
def crossover(p1, p2, r_cross):

    # 1. Create offsprings as exact copies of the parents
    c1 = p1.copy()
    c2 = p2.copy()

    # 2. Roll the dice to see if crossover happen
    if np.random.rand() < r_cross:

        # 3. Pick a random split point somewhere in the middle of the DNA
        pt = np.random.randint(1, len(p1) - 1)

        # 4. Single point crossover
        c1 = p1[:pt] + p2[pt:]
        c2 = p2[:pt] + p1[pt:]
    
    return [c1,c2]

### Step 6: Mutation
To prevent the algorithm from stagnating, we introduce spontaneous variation through **Bit-Flip Mutation**. For every single bit in an offspring's chromosome, there is a small probability (`r_mut`) that it will flip from 0 to 1, or 1 to 0. This ensures continuous exploration of the parameter search space.

In [6]:
def mutate(bitstring, r_mut):
    # Loop through every single bit in the 24-bits chromosome
    for i in range(len(bitstring)):

        if np.random.rand() < r_mut:
            bitstring[i] = 1 - bitstring[i]

### Step 7: The Master Evolutionary Loop
This function acts as the central orchestrator. It iteratively evaluates fitness, tracks the elite solutions globally, selects parents, and breeds a completely new population over `n_iter` generations until the optimal PID gains are discovered.

In [7]:
def genetic_optimize(obj_fx, bounds, n_bits, n_pop, n_iter, r_cross, r_mut):

    # Initialize population
    total_bits_per_drone = n_bits * len(bounds)
    population = [np.random.randint(0, 2, total_bits_per_drone).tolist() for _ in range(n_pop)]

    # Keep track of scores
    best_score_overall = 0.0
    best_solution_overall = None
    best_decoded_overall = None

    for gen in range(n_iter):

        # Decode population
        decoded = [decode(bounds, n_bits, p) for p in population]

        # Evaluate fitness
        scores = [obj_fx(d) for d in decoded]

        # Find best drone (chromosome) for current population
        best_idx_gen = scores.index(max(scores))
        current_best_score = scores[best_idx_gen]
        current_best_decoded = decoded[best_idx_gen]

        # Update overall best if current generation found a better one
        if current_best_score > best_score_overall:
            best_score_overall = current_best_score
            best_solution_overall = population[best_idx_gen]
            best_decoded_overall = decoded[best_idx_gen]
            print(f'> Generation {gen}: New Best Score = {best_score_overall:.4f} | Kp, Ki, Kd = {[round(x, 3) for x in best_decoded_overall]}')

        # NEW GENERATION    
        new_population = []
        while len(new_population) < n_pop:
            p1 = selection(population, scores)
            p2 = selection(population, scores)

            c1, c2 = crossover(p1, p2, r_cross)

            mutate(c1, r_mut)
            mutate(c2, r_mut)

            # Add new offspring to the new population
            new_population.extend([c1,c2])
        
        # Replace old population with the new one
        population = new_population[:n_pop]
    
    print(f"\n--- OPTIMIZATION COMPLETE ---")
    print(f"Final Best Kp, Ki, Kd: {[round(x, 4) for x in best_decoded_overall]}")
    print(f"Final Best Fitness Score: {best_score_overall:.4f}")

    return best_decoded_overall

### Execution and Hyperparameter Configuration
In this final block, we define the search space boundaries representing the physical constraints of our specific UAV system. We then establish the fundamental hyperparameters governing the evolutionary process and execute the optimization routine.

In [8]:
if __name__ == "__main__":

    # Define the bounds (rules/limits)
    drone_bounds = [
        [0.0, 10.0], # Kp
        [0.0, 2.0],  # Ki
        [0.0, 2.0]   # Kd
    ]

    # Algorithm hyperparameter
    BITS = 8
    POPULATION_SIZE = 50 
    GENERATIONS = 100    
    CROSSOVER_RATE = 0.9  # 90% chance to crossover
    MUTATION_RATE = 0.05  # 5% chance to mutate a bit

    best_gains = genetic_optimize(drone_objective, drone_bounds, BITS, POPULATION_SIZE, GENERATIONS, CROSSOVER_RATE, MUTATION_RATE)

> Generation 0: New Best Score = 0.0098 | Kp, Ki, Kd = [9.18, 0.094, 0.477]
> Generation 2: New Best Score = 0.0099 | Kp, Ki, Kd = [9.18, 0.031, 0.352]
> Generation 5: New Best Score = 0.0099 | Kp, Ki, Kd = [9.492, 0.062, 0.484]
> Generation 6: New Best Score = 0.0099 | Kp, Ki, Kd = [9.492, 0.031, 0.43]
> Generation 8: New Best Score = 0.0099 | Kp, Ki, Kd = [9.492, 0.0, 0.289]
> Generation 10: New Best Score = 0.0099 | Kp, Ki, Kd = [9.961, 0.0, 0.156]
> Generation 11: New Best Score = 0.0099 | Kp, Ki, Kd = [9.961, 0.0, 0.141]
> Generation 12: New Best Score = 0.0099 | Kp, Ki, Kd = [9.883, 0.0, 0.141]
> Generation 13: New Best Score = 0.0099 | Kp, Ki, Kd = [9.883, 0.016, 0.008]
> Generation 15: New Best Score = 0.0099 | Kp, Ki, Kd = [9.922, 0.008, 0.023]
> Generation 16: New Best Score = 0.0099 | Kp, Ki, Kd = [9.961, 0.0, 0.031]
> Generation 38: New Best Score = 0.0099 | Kp, Ki, Kd = [9.961, 0.0, 0.023]

--- OPTIMIZATION COMPLETE ---
Final Best Kp, Ki, Kd: [9.9609, 0.0, 0.0234]
Final Be